In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def melon_artist_tracks():
    url = "https://www.melon.com/chart/index.htm"

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Referer": "https://www.melon.com/chart/index.htm"
    }

    response = requests.get(url, headers=headers)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    rows = soup.select("tr.lst50, tr.lst100")
    
    like_url = "https://www.melon.com/commonlike/getSongLike.json"

    data = []

    for row in rows:
        song_id = row["data-song-no"]

        rank_tag = row.select_one("span.rank")
        title_tag = row.select_one("div.ellipsis.rank01 a")
        artist_tag = row.select_one("div.ellipsis.rank02 a")
        album_tag = row.select_one("div.ellipsis.rank03 a")

        if not title_tag:
            continue

        rank = rank_tag.get_text(strip=True)
        title = title_tag.get_text(strip=True)
        artist = artist_tag.get_text(strip=True) if artist_tag else ""
        album = album_tag.get_text(strip=True) if album_tag else ""

        like_response = requests.get(
            like_url,
            headers=headers,
            params={"contsIds": song_id} 
        )
        like_data = like_response.json()
        likes = like_data["contsLike"][0]["SUMMCNT"]

        data.append({
            "순위": rank +'위',
            "곡제목": title,
            "가수명": artist,
            "앨범명": album,
            "좋아요": likes
        })

    df = pd.DataFrame(data)
    df.index = df.index + 1

    file_name = "melon_chart_100.csv"
    df.to_csv(file_name, encoding="utf-8-sig")

    return df

melon_artist_tracks()